This script removes the expression (zeroes it) of genes that were recurrently undetected in published cfRNA cohorts, ranging frop top 0 to 100%. After removal, the data is still CPM normalised.

Input:
1) Counts file of the simulutated samples:
    - 1000 samples with random contributions (20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt) generated by the 20250618_SimulatedDatasets_v2_NoDup_1000/00_Random-Sampling_Simulated_v2_1000.py script


2) List of genes that are repeatedly missing or lowly expressed in cfRNA datasets (20260502_Genes_NotDetected_2of5-Cohorts_CPM10_5Percent.csv) generated by the 20250514_Published-cfRNA/08_cfRNA-Gene-Detection.ipynb script

In [2]:
import os
import pandas as pd
import numpy as np

# Paths
counts_file = "Data/20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt"
gene_list_file = "Data/20260502_Genes_NotDetected_2of5-Cohorts_CPM10_5Percent.csv"
gtf_path = "gencode.v46.annotation.gtf"
out_dir = "Data"

removal_fractions = [0.20, 0.40, 0.60, 0.80, 1.00]
random_seed = 42


#############################################
# Load GENCODE map
#############################################

gtf = pd.read_csv(gtf_path, sep="\t", comment="#", header=None, low_memory=False)
gtf.columns = [
    "chr", "source", "feature", "start", "end",
    "score", "strand", "frame", "attribute"
]

genes_gtf = gtf[gtf["feature"] == "gene"].copy()

def extract_attribute(attr_str, key):
    for field in str(attr_str).split(";"):
        field = field.strip()
        if field.startswith(key):
            return field.split('"')[1]
    return None

genes_gtf["gene_id"] = genes_gtf["attribute"].apply(lambda x: extract_attribute(x, "gene_id"))
genes_gtf["gene_name"] = genes_gtf["attribute"].apply(lambda x: extract_attribute(x, "gene_name"))
genes_gtf["ensembl_id"] = genes_gtf["gene_id"].astype(str).str.split(".").str[0]

gencode_map = genes_gtf[["gene_name", "ensembl_id"]].dropna().drop_duplicates()

ensembl_to_gene_name = dict(zip(gencode_map["ensembl_id"], gencode_map["gene_name"]))


#############################################
# Load counts and gene list
#############################################

counts_df = pd.read_csv(counts_file, sep="\t", index_col=0)
counts_df.index = counts_df.index.astype(str)

gene_df = pd.read_csv(gene_list_file)

counts_gene_set = set(counts_df.index)

# Direct gene-name match
gene_df["direct_gene_name_match"] = gene_df["gene_name"].astype(str).isin(counts_gene_set)

# Fallback via Ensembl ID -> GENCODE gene name
gene_df["gencode_gene_name_from_ensembl"] = (
    gene_df["gencode_ensembl_id"].astype(str).map(ensembl_to_gene_name)
)

gene_df["fallback_gene_name_match"] = (
    gene_df["gencode_gene_name_from_ensembl"].astype(str).isin(counts_gene_set)
)

# Final matched Geneid in counts matrix
gene_df["matched_geneid"] = pd.NA

gene_df.loc[
    gene_df["direct_gene_name_match"],
    "matched_geneid"
] = gene_df.loc[
    gene_df["direct_gene_name_match"],
    "gene_name"
]

gene_df.loc[
    ~gene_df["direct_gene_name_match"] & gene_df["fallback_gene_name_match"],
    "matched_geneid"
] = gene_df.loc[
    ~gene_df["direct_gene_name_match"] & gene_df["fallback_gene_name_match"],
    "gencode_gene_name_from_ensembl"
]

matched_genes = (
    gene_df["matched_geneid"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print("\nAlignment summary")
print(f"Genes in removal list: {len(gene_df)}")
print(f"Matched by gene name: {gene_df['direct_gene_name_match'].sum()}")
print(
    "Matched by Ensembl fallback: "
    f"{((~gene_df['direct_gene_name_match']) & gene_df['fallback_gene_name_match']).sum()}"
)
print(f"Total unique matched genes in counts matrix: {len(matched_genes)}")


#############################################
# Remove fractions and CPM-normalize
#############################################

rng = np.random.default_rng(random_seed)
matched_genes_shuffled = matched_genes.copy()
rng.shuffle(matched_genes_shuffled)

for frac in removal_fractions:
    n_remove = int(round(frac * len(matched_genes_shuffled)))
    genes_to_remove = matched_genes_shuffled[:n_remove]

    modified_df = counts_df.copy()
    modified_df.loc[genes_to_remove, :] = 0

    library_sizes = modified_df.sum(axis=0).replace(0, pd.NA)
    cpm_df = modified_df.div(library_sizes, axis=1) * 1e6
    cpm_df = cpm_df.fillna(0)

    pct = int(frac * 100)

    output_path = (
        f"{out_dir}/20250616_All-Tissues-NoDup_Random_{pct}_percent_removed_CPM.txt"
    )

    cpm_df.to_csv(output_path, sep="\t")

    print(
        f"{pct}% removal: zeroed {n_remove} genes; "
        f"saved {output_path}"
    )

print("\nProcessing complete.")


Alignment summary
Genes in removal list: 50972
Matched by gene name: 50972
Matched by Ensembl fallback: 0
Total unique matched genes in counts matrix: 49689
20% removal: zeroed 9938 genes; saved Data/20250616_All-Tissues-NoDup_Random_20_percent_removed_CPM.txt
40% removal: zeroed 19876 genes; saved Data/20250616_All-Tissues-NoDup_Random_40_percent_removed_CPM.txt
60% removal: zeroed 29813 genes; saved Data/20250616_All-Tissues-NoDup_Random_60_percent_removed_CPM.txt
80% removal: zeroed 39751 genes; saved Data/20250616_All-Tissues-NoDup_Random_80_percent_removed_CPM.txt
100% removal: zeroed 49689 genes; saved Data/20250616_All-Tissues-NoDup_Random_100_percent_removed_CPM.txt

Processing complete.
